# Análise de pedidos do delivery

Este notebook explora os pedidos, trata valores ausentes, cria a receita por item, analisa vendas por item e mês, integra os dados do cardápio e calcula KPIs do negócio.

Os arquivos `pedidos.csv` e `cardapio.csv` devem estar na raiz do projeto. Execute as células em ordem.

## 1. Bibliotecas e carregamento dos dados

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")

diretorio_dados = Path.cwd()
caminho_pedidos = diretorio_dados / "pedidos.csv"
caminho_cardapio = diretorio_dados / "cardapio.csv"

if not caminho_pedidos.exists() or not caminho_cardapio.exists():
    raise FileNotFoundError(
        "Os arquivos pedidos.csv e cardapio.csv precisam estar no diretório "
        f"de execução do notebook: {diretorio_dados}"
    )

pedidos_brutos = pd.read_csv(caminho_pedidos)
cardapio = pd.read_csv(caminho_cardapio)

print(f"Pedidos: {pedidos_brutos.shape[0]:,} linhas × {pedidos_brutos.shape[1]} colunas")
print(f"Cardápio: {cardapio.shape[0]:,} linhas × {cardapio.shape[1]} colunas")

## 2. Análise exploratória (EDA)

Inspecionamos as primeiras e últimas linhas, os tipos, o número de registros e as estatísticas descritivas antes do tratamento.

In [ ]:
display(pedidos_brutos.head())
display(pedidos_brutos.tail())
pedidos_brutos.info()
display(pedidos_brutos.describe(include="all").T)

print("\nTipos de dados:")
display(pedidos_brutos.dtypes.rename("tipo").to_frame())
print(f"Registros: {len(pedidos_brutos):,}")
print(f"Colunas: {pedidos_brutos.shape[1]}")

## 3. Qualidade dos dados e preparação

A quantidade ausente é preenchida pela média observada. Linhas sem preço unitário são removidas, pois não é possível calcular sua receita. A conversão de campos numéricos e de datas é validada; datas inválidas serão excluídas somente da análise temporal.

In [ ]:
print("Valores nulos por coluna (dados originais):")
display(pedidos_brutos.isna().sum().rename("valores_nulos").to_frame())
print("\nDuplicatas exatas nos pedidos:", pedidos_brutos.duplicated().sum())
print("Itens no cardápio sem nome:", cardapio["Item"].isna().sum())

pedidos = pedidos_brutos.copy()

for coluna in ["Quantidade", "Preco_Unitario"]:
    pedidos[coluna] = pd.to_numeric(pedidos[coluna], errors="coerce")

quantidade_nula_inicial = pedidos["Quantidade"].isna().sum()
preco_nulo_inicial = pedidos["Preco_Unitario"].isna().sum()

if pedidos["Quantidade"].notna().sum() == 0:
    raise ValueError("Não há valores válidos em Quantidade para calcular a média.")

media_quantidade = pedidos["Quantidade"].mean()
pedidos["Quantidade"] = pedidos["Quantidade"].fillna(media_quantidade)
pedidos = pedidos.dropna(subset=["Preco_Unitario"]).copy()

pedidos["Data"] = pd.to_datetime(pedidos["Data"], errors="coerce")
datas_invalidas = pedidos["Data"].isna().sum()

print(f"Quantidade ausente/inválida preenchida com a média ({media_quantidade:.2f}): {quantidade_nula_inicial}")
print(f"Linhas removidas por preço unitário ausente/inválido: {preco_nulo_inicial}")
print(f"Datas ausentes/inválidas (serão ignoradas apenas na análise temporal): {datas_invalidas}")
print(f"Linhas restantes para análise: {len(pedidos):,}")

pedidos["Receita_Item"] = pedidos["Quantidade"] * pedidos["Preco_Unitario"]
display(pedidos.head())
print("\nValores nulos após o tratamento:")
display(pedidos.isna().sum().rename("valores_nulos").to_frame())

## 4. Vendas e receita por item

Os rankings consideram a quantidade total vendida e a receita total calculada (`Quantidade × Preco_Unitario`).

In [ ]:
vendas_por_item = (
    pedidos.groupby("Item", as_index=False)
    .agg(Quantidade_Total=("Quantidade", "sum"), Receita_Total=("Receita_Item", "sum"))
    .sort_values("Receita_Total", ascending=False)
)

print("Resumo de vendas por item:")
display(vendas_por_item)
print("Top 5 itens por quantidade vendida:")
display(vendas_por_item.nlargest(5, "Quantidade_Total")[ ["Item", "Quantidade_Total", "Receita_Total"] ])
print("Top 5 itens por receita:")
display(vendas_por_item.nlargest(5, "Receita_Total")[ ["Item", "Quantidade_Total", "Receita_Total"] ])

## 5. Análise temporal

Datas inválidas ficam de fora apenas desta etapa. A receita é agregada por mês-calendário e apresentada em ordem cronológica.

In [ ]:
pedidos_temporais = pedidos.dropna(subset=["Data"]).copy()
pedidos_temporais["Mes"] = pedidos_temporais["Data"].dt.to_period("M")

receita_mensal = (
    pedidos_temporais.groupby("Mes", as_index=False)["Receita_Item"]
    .sum()
    .rename(columns={"Receita_Item": "Receita_Total"})
    .sort_values("Mes")
)
display(receita_mensal)

if not receita_mensal.empty:
    plt.figure(figsize=(12, 5))
    plt.plot(receita_mensal["Mes"].astype(str), receita_mensal["Receita_Total"], marker="o")
    plt.title("Receita mensal")
    plt.xlabel("Mês")
    plt.ylabel("Receita")
    plt.xticks(rotation=60, ha="right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Não há datas válidas para construir a série temporal.")

## 6. Integração com o cardápio e análise por categoria

O merge é do tipo `left` para preservar todos os pedidos. Itens sem correspondência no cardápio são identificados; eles não entram em categorias conhecidas.

In [ ]:
if cardapio["Item"].duplicated().any():
    raise ValueError("O cardápio contém nomes de Item duplicados; revise antes do merge para evitar duplicar pedidos.")

pedidos_cardapio = pedidos.merge(
    cardapio[["Item", "Categoria", "Preco_Base"]],
    on="Item",
    how="left",
    validate="many_to_one",
    indicator=True,
)

itens_sem_categoria = pedidos_cardapio.loc[pedidos_cardapio["_merge"] == "left_only", "Item"].drop_duplicates()
print(f"Itens de pedido sem correspondência no cardápio: {itens_sem_categoria.tolist()}")

receita_por_categoria = (
    pedidos_cardapio.groupby("Categoria", as_index=False, dropna=False)["Receita_Item"]
    .sum()
    .rename(columns={"Receita_Item": "Receita_Total"})
    .sort_values("Receita_Total", ascending=False)
)
display(receita_por_categoria)

categorias_conhecidas = receita_por_categoria.dropna(subset=["Categoria"])
if not categorias_conhecidas.empty:
    lider_categoria = categorias_conhecidas.iloc[0]
    print(f"Categoria com maior receita: {lider_categoria['Categoria']} ({lider_categoria['Receita_Total']:.2f})")
else:
    print("Não há categorias correspondentes para identificar uma líder.")

## 7. Filtro: Salgados com quantidade acima de 10

O filtro usa a categoria obtida no merge com o cardápio.

In [ ]:
pedidos_salgados_maiores_10 = pedidos_cardapio.loc[
    (pedidos_cardapio["Categoria"] == "Salgados")
    & (pedidos_cardapio["Quantidade"] > 10)
].drop(columns="_merge")

print(f"Pedidos de Salgados com quantidade superior a 10: {len(pedidos_salgados_maiores_10):,}")
display(pedidos_salgados_maiores_10)

## 8. KPIs e percentis com NumPy

O **ticket médio** é definido aqui como receita total dividida pelo número de `ID_Pedido` únicos. Essa definição pressupõe que cada identificador represente um pedido; se o arquivo estiver em nível de item, essa medida representa a receita média por pedido agrupado, e não por linha.

In [ ]:
receita_total = pedidos["Receita_Item"].sum()
total_itens_vendidos = pedidos["Quantidade"].sum()
numero_pedidos = pedidos["ID_Pedido"].nunique()
ticket_medio = receita_total / numero_pedidos if numero_pedidos else np.nan

print(f"Receita total: {receita_total:,.2f}")
print(f"Total de itens vendidos: {total_itens_vendidos:,.2f}")
print(f"Número de pedidos únicos: {numero_pedidos:,}")
print(f"Ticket médio por pedido: {ticket_medio:,.2f}")

percentis = [25, 50, 75]
for coluna in ["Preco_Unitario", "Quantidade"]:
    valores_validos = pedidos[coluna].dropna().to_numpy()
    if valores_validos.size:
        resultados = np.percentile(valores_validos, percentis)
        print(f"\nPercentis de {coluna}:")
        display(pd.Series(resultados, index=["25%", "50%", "75%"], name=coluna).to_frame())
    else:
        print(f"Não há valores válidos em {coluna} para calcular percentis.")

## 9. Resumo executivo

Os resultados acima permitem comparar os itens de maior volume e faturamento, acompanhar a evolução mensal da receita, identificar a categoria líder e monitorar receita, volume e ticket médio. Antes de decisões comerciais, considere investigar datas inválidas, itens sem categoria e valores imputados.